# 🤟 ASL Alphabet — YOLOv8m Local Training
**OS:** Ubuntu / Linux &nbsp;|&nbsp; **GPU:** NVIDIA CUDA &nbsp;|&nbsp; **Model:** YOLOv8m

| Step | Description |
|------|-------------|
| 1 | Check GPU & install dependencies |
| 2 | Verify dataset structure |
| 3 | Write `data.yaml` |
| 4 | Train YOLOv8m |
| 5 | Evaluate results |
| 6 | Visualize predictions |
| 7 | Export to ONNX |

---
## 📦 Step 1 — Install Dependencies

In [2]:
import subprocess, sys

# ── Detect CUDA version ────────────────────────────────────
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise EnvironmentError('❌ nvidia-smi not found. Check NVIDIA drivers: sudo apt install nvidia-driver-535')

print(result.stdout)

import re
match = re.search(r'CUDA Version: (\d+)\.(\d+)', result.stdout)
cuda_major = int(match.group(1)) if match else 0
cuda_minor = int(match.group(2)) if match else 0
print(f'✅ CUDA {cuda_major}.{cuda_minor} detected')

Sat May  9 14:37:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 Ti     On  |   00000000:01:00.0 Off |                  N/A |
| N/A   48C    P8              2W /   50W |      65MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ── Install PyTorch with correct CUDA wheels ───────────────
if cuda_major >= 12:
    whl = 'https://download.pytorch.org/whl/cu121'
    print('Installing PyTorch for CUDA 12.x ...')
elif cuda_major == 11 and cuda_minor >= 8:
    whl = 'https://download.pytorch.org/whl/cu118'
    print('Installing PyTorch for CUDA 11.8 ...')
else:
    whl = None
    print('⚠️  Old CUDA — installing CPU PyTorch (slow!)')

if whl:
    !{sys.executable} -m pip install torch torchvision torchaudio --index-url {whl} -q
else:
    !{sys.executable} -m pip install torch torchvision torchaudio -q

Installing PyTorch for CUDA 12.x ...
/bin/bash: line 1: /home/abdelrahman/Downloads/SUTECH/third: No such file or directory


In [5]:
# ── Install remaining packages ─────────────────────────────
!{sys.executable} -m pip install \
    ultralytics>=8.0.0 \
    opencv-python>=4.8.0 \
    pyyaml>=6.0 \
    pillow>=10.0.0 \
    numpy>=1.24.0 \
    matplotlib>=3.7.0 \
    seaborn pandas tqdm -q

print('✅ All packages installed')

/bin/bash: line 1: /home/abdelrahman/Downloads/SUTECH/third: No such file or directory
✅ All packages installed


In [6]:
# ── Verify GPU ─────────────────────────────────────────────
import torch
import ultralytics
import cv2
import numpy as np

print('='*50)
print('  System Info')
print('='*50)
print(f'  PyTorch      : {torch.__version__}')
print(f'  CUDA avail   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU          : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM         : {vram:.1f} GB')
    # Recommend batch size
    BATCH = 64 if vram >= 16 else 32 if vram >= 8 else 16 if vram >= 4 else 8
    print(f'  Recommended batch size: {BATCH}')
else:
    BATCH = 8
    print('  ⚠️  No GPU — using CPU (very slow)')
print(f'  Ultralytics  : {ultralytics.__version__}')
print(f'  OpenCV       : {cv2.__version__}')
print('='*50)

  System Info
  PyTorch      : 2.11.0+cu130
  CUDA avail   : True
  GPU          : NVIDIA GeForce GTX 1650 Ti
  VRAM         : 3.9 GB
  Recommended batch size: 8
  Ultralytics  : 8.4.46
  OpenCV       : 4.11.0


---
## ⚙️ Step 2 — Configuration

In [8]:
from pathlib import Path

# ── UPDATE THIS PATH ───────────────────────────────────────
DATASET_PATH = './asl_yolo'       # output folder from convert_to_yolo.py
RUNS_DIR     = './runs'

# ── Training hyperparameters ───────────────────────────────
EPOCHS    = 50
PATIENCE  = 10       # early stopping
IMG_SIZE  = 200      # ASL images are 200×200
WORKERS   = 8
DEVICE    = 0        # GPU index (0 = first GPU)

CLASSES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z',
    'del','nothing','space'
]

print(f'📁 Dataset : {Path(DATASET_PATH).resolve()}')
print(f'🏷️  Classes : {len(CLASSES)}')
print(f'🔢 Batch   : {BATCH}  (auto from VRAM)')
print(f'📐 ImgSize : {IMG_SIZE}')

📁 Dataset : /home/abdelrahman/Downloads/SUTECH/third year second semester/image /project/asl_yolo
🏷️  Classes : 29
🔢 Batch   : 8  (auto from VRAM)
📐 ImgSize : 200


---
## 🗂️ Step 3 — Verify Dataset

In [9]:
import os

root = Path(DATASET_PATH)
print('Dataset structure check:\n')

all_ok = True
for split in ('train', 'val'):
    img_dir = root / 'images' / split
    lbl_dir = root / 'labels' / split
    n_imgs  = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbls  = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    status  = '✅' if n_imgs > 0 and n_imgs == n_lbls else '❌'
    print(f'  {status} {split:6s}  images={n_imgs:5d}  labels={n_lbls:5d}')
    if n_imgs == 0:
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        f'\n❌ Dataset missing at: {root.resolve()}\n'
        '   Run convert_to_yolo.py first!'
    )
else:
    print('\n✅ Dataset OK!')

Dataset structure check:

  ✅ train   images= 4760  labels= 4760
  ✅ val     images=  840  labels=  840

✅ Dataset OK!


---
## 📝 Step 4 — Write data.yaml

In [10]:
import yaml

yaml_path = str(root / 'data.yaml')

data_yaml = {
    'path'  : str(root.resolve()),
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : len(CLASSES),
    'names' : CLASSES,
}

with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f'✅ data.yaml → {yaml_path}\n')
print(open(yaml_path).read())

✅ data.yaml → asl_yolo/data.yaml

path: /home/abdelrahman/Downloads/SUTECH/third year second semester/image /project/asl_yolo
train: images/train
val: images/val
nc: 29
names:
- A
- B
- C
- D
- E
- F
- G
- H
- I
- J
- K
- L
- M
- N
- O
- P
- Q
- R
- S
- T
- U
- V
- W
- X
- Y
- Z
- del
- nothing
- space



---
## 🚀 Step 5 — Train YOLOv8m

In [11]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')   # downloads pretrained weights (~52 MB)

results = model.train(
    data          = yaml_path,
    epochs        = EPOCHS,
    patience      = PATIENCE,
    imgsz         = IMG_SIZE,
    batch         = BATCH,
    workers       = WORKERS,
    device        = DEVICE,

    # Optimizer
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,

    # Augmentation
    augment       = True,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    fliplr        = 0.0,     # disabled — flipping changes ASL meaning!
    degrees       = 10,
    translate     = 0.1,
    scale         = 0.3,

    # Output
    project       = RUNS_DIR,
    name          = 'asl_yolov8m',
    exist_ok      = True,
    verbose       = True,
    plots         = True,
    save          = True,
    save_period   = 10,
)

BEST_WEIGHTS = f'{RUNS_DIR}/asl_yolov8m/weights/best.pt'
print(f'\n✅ Training done! Best weights → {BEST_WEIGHTS}')

New https://pypi.org/project/ultralytics/8.4.48 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.46 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 3716MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=asl_yolo/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=asl

---
## 📊 Step 6 — Evaluate

In [ ]:
best_model = YOLO(BEST_WEIGHTS)
metrics    = best_model.val(data=yaml_path, imgsz=IMG_SIZE, device=DEVICE)

print('\n📈 Validation Results:')
print(f'   mAP50      : {metrics.box.map50:.4f}')
print(f'   mAP50-95   : {metrics.box.map:.4f}')
print(f'   Precision  : {metrics.box.mp:.4f}')
print(f'   Recall     : {metrics.box.mr:.4f}')

---
## 📈 Step 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

results_dir = f'{RUNS_DIR}/asl_yolov8m'

plots = {
    'results.png'          : 'Loss & mAP Curves',
    'confusion_matrix.png' : 'Confusion Matrix',
    'PR_curve.png'         : 'Precision-Recall Curve',
}

for filename, title in plots.items():
    path = os.path.join(results_dir, filename)
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print(f'⚠️  {filename} not found yet')

---
## 🖼️ Step 8 — Visualize Predictions

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2

# Pick random val images
val_imgs = list((root / 'images' / 'val').glob('*.jpg'))
sample   = random.sample(val_imgs, min(12, len(val_imgs)))

preds = best_model.predict(
    source   = [str(p) for p in sample],
    imgsz    = IMG_SIZE,
    conf     = 0.5,
    save     = True,
    project  = './predictions',
    name     = 'val_samples',
    exist_ok = True,
)

# Display grid
pred_imgs = sorted(glob.glob('./predictions/val_samples/*.jpg'))[:12]
cols, rows = 4, 3
fig, axes = plt.subplots(rows, cols, figsize=(16, 12))
fig.suptitle('YOLOv8m Predictions on Validation Set', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(pred_imgs):
        img = cv2.cvtColor(cv2.imread(pred_imgs[i]), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(Path(pred_imgs[i]).stem[:20], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 💾 Step 9 — Export to ONNX

In [ ]:
best_model.export(
    format   = 'onnx',
    imgsz    = IMG_SIZE,
    dynamic  = False,
    simplify = True,
)

onnx_path = BEST_WEIGHTS.replace('.pt', '.onnx')
print(f'✅ ONNX exported → {onnx_path}')

---
## 📋 Summary

| Item | Value |
|------|-------|
| Model | YOLOv8m |
| Classes | 29 (A–Z + del + nothing + space) |
| Images/class | 200 (50 × 4 ranges) |
| Image size | 200 × 200 |
| Epochs | 50 (early stop @ patience=10) |
| Optimizer | AdamW + Cosine LR |
| Best weights | `runs/asl_yolov8m/weights/best.pt` |
| ONNX export | `runs/asl_yolov8m/weights/best.onnx` |